# 04. Penanganan Data yang Hilang (Handling Missing Data)

## Prinsip
Identifikasi terlebih dahulu nilai kosong asli dan penanda tekstual yang benar-benar digunakan sumber. Penanda seperti `---` tidak otomatis berarti nol. Jangan melakukan imputasi atau eliminasi tanpa alasan yang dapat dipertanggungjawabkan.

Analisis MCAR/MAR/MNAR membutuhkan bukti dan pemahaman proses pembentukan data; pola statistik saja tidak selalu cukup untuk menetapkan mekanismenya.

In [ ]:
from pathlib import Path
import pandas as pd
from eda_knkt.config import PROCESSED_DATA_DIR
from eda_knkt.ingestion import load_csv
from eda_knkt.cleaning import replace_placeholders
from eda_knkt.transformation import convert_numeric_columns

DATA_FILE = PROCESSED_DATA_DIR / 'data_logger_knkt_tabel_6_10.csv'
df_raw = load_csv(DATA_FILE)
df_missing = replace_placeholders(df_raw, ['---'])
NUMERIC_OPERATIONAL_COLUMNS = [
    'km', 'm', 'Speed (km/h)', 'Acceleration (m/sec²)',
    'Load (amps)', 'EAB BP (psi)', 'EAB BC (psi)'
]
df_transformed = convert_numeric_columns(
    df_missing, NUMERIC_OPERATIONAL_COLUMNS, errors='coerce'
)
df_transformed['Timestamp'] = pd.to_datetime(
    df_transformed['Date'].astype(str) + ' ' + df_transformed['Time'].astype(str),
    format='%d/%m/%Y %H.%M.%S', errors='raise'
)
df_transformed = df_transformed.sort_values('Timestamp').reset_index(drop=True)
df_transformed['Elapsed (s)'] = (
    df_transformed['Timestamp'] - df_transformed['Timestamp'].iloc[0]
).dt.total_seconds()
print(f'Loaded {len(df_raw)} records × {len(df_raw.columns)} columns from {DATA_FILE.name}')

In [ ]:
import pandas as pd
from eda_knkt.cleaning import missing_value_summary, replace_placeholders

# Prasyarat: df_raw telah dimuat pada notebook 03.
if "df_raw" not in globals():
    print("Jalankan notebook 03 dan muat dataset terlebih dahulu.")
else:
    # Gunakan placeholder hanya setelah diverifikasi pada dokumen sumber.
    VERIFIED_PLACEHOLDERS = ["---"]
    df_missing = replace_placeholders(df_raw, VERIFIED_PLACEHOLDERS)
    missing_report = missing_value_summary(df_missing)
    display(missing_report.sort_values("missing_percent", ascending=False))
    print("Total missing cells:", int(df_missing.isna().sum().sum()))

## Keputusan penanganan
Untuk setiap kolom yang memiliki nilai hilang, dokumentasikan:
1. Penanda yang ditemukan dan bukti dari sumber.
2. Jumlah serta persentase nilai hilang.
3. Apakah nilai hilang memiliki makna operasional/teknis.
4. Alasan memilih mempertahankan, mengecualikan, atau mengimputasi.
5. Dampak keputusan terhadap jumlah observasi.

Jangan memilih MCAR, MAR, atau MNAR tanpa dasar. Imputasi mean/median/mode, KNN, atau MICE tidak otomatis tepat untuk rekaman kejadian teknis yang memiliki urutan waktu dan makna domain.

In [ ]:
# Pemeriksaan duplikasi dilakukan terpisah dari keputusan penghapusan.
if "df_missing" in globals():
    print("Exact duplicate rows:", int(df_missing.duplicated().sum()))
    print("Rows:", len(df_missing))
    # Jangan hapus duplikasi sebelum memastikan aturan identitas rekaman/event.